# Heston-Arb Edge Diagnostic (Colab / T4)

Runs the **gap-decomposition** test: for each option contract tracked across consecutive
sessions, it splits the change in `gap = model_iv - market_iv` into

- **market_contrib** = `sign(gap_t) * d_market` -- the ONLY part that earns P&L (market moving toward the model)
- **model_contrib**  = `sign(gap_t) * (-d_model)` -- earns NOTHING (the model re-fitting toward the market)

If gaps close mainly via **model_contrib**, the strategy is structurally dead (the model chases the market).

Data = real SPY trade bars from Alpaca (no historical bid/ask available on the plan), so this measures
convergence DIRECTION, not net-of-cost P&L. It can KILL the thesis; it cannot prove a durable edge.

**Requirements (Colab Secrets -- key icon, left sidebar):** `GITHUB_TOKEN`, `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`, optional `FRED_API_KEY`. Set Runtime -> Change runtime type -> T4 GPU.

In [ ]:
# Cell 1: dependencies (diagnostic path only -- no cvxpy needed)
!pip install -q -U "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q optax diffrax scipy requests pandas

In [ ]:
# Cell 2: clone / update the repo (needs GITHUB_TOKEN in Colab Secrets)
import os, subprocess, sys, shutil
from google.colab import userdata

REPO = '/content/heston-arb'
token = userdata.get('GITHUB_TOKEN')
clone_url = f'https://{token}@github.com/AliAlpOezer/heston-arb.git'

if not os.path.exists(os.path.join(REPO, '.git')):
    if os.path.exists(REPO):
        shutil.rmtree(REPO)
    r = subprocess.run(['git', 'clone', clone_url, REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed: {r.stderr.strip()}')
    print('Cloned.')
else:
    r = subprocess.run(['git', '-C', REPO, 'pull'], capture_output=True, text=True)
    print('Pulled:', r.stdout.strip() or r.stderr.strip())

if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)
print('Repo ready at', REPO)

In [ ]:
# Cell 3: API keys + GPU check
import os
from google.colab import userdata
os.environ['ALPACA_API_KEY']    = userdata.get('ALPACA_API_KEY')
os.environ['ALPACA_SECRET_KEY'] = userdata.get('ALPACA_SECRET_KEY')
try:
    os.environ['FRED_API_KEY'] = userdata.get('FRED_API_KEY') or ''
except Exception:
    os.environ['FRED_API_KEY'] = ''

import jax
print('JAX devices:', jax.devices())  # expect a cuda/gpu device on T4

In [ ]:
# Cell 4: SMOKE TEST (1 month) -- confirms data + pipeline run end-to-end, ~1-2 min
from backtest.gap_decomposition import run
summ = run('SPY', '2024-06-03', '2024-06-28', n_steps=200, csv_path='/content/gap_smoke.csv')

In [ ]:
# Cell 5: FULL RUN (2024-06 .. 2025-12). On T4 this should be minutes, not hours.
from backtest.gap_decomposition import run
summ = run('SPY', '2024-06-01', '2025-12-31', n_steps=300, csv_path='/content/gap_full.csv')

In [ ]:
# Cell 6: inspect raw per-pair rows + download
import pandas as pd
df = pd.read_csv('/content/gap_full.csv')
print('rows:', len(df), '| gap points:', int(df.is_gap_point.sum()), '| tradeable signals:', int(df.is_signal.sum()))
print('\nmean RMSE on signal days:', round(df['rmse_t'].mean(), 4), '(gate is 0.10)')
print('\nGap-point decomposition (vol pts/day):')
g = df[df.is_gap_point]
print('  market_contrib mean:', round(g.market_contrib.mean()*100, 3), 'vp')
print('  model_contrib  mean:', round(g.model_contrib.mean()*100, 3), 'vp')
from google.colab import files
files.download('/content/gap_full.csv')